In [0]:
from pyspark.sql.functions import (
    col, to_json, from_json, round, to_date, year,
    regexp_replace, max, when, split, explode, trim,
    lower, avg, count, sum   
)
from pyspark.sql.types import MapType, StringType, LongType

Load Data

In [0]:
src = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"

df_raw = (
    spark.read
    .option("multiline", "true")
    .option("mode", "PERMISSIVE")
    .json(src)
)

display(df_raw.limit(5))
df_raw.printSchema()

data,id
"List(10, List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP), 13990, Valve, 0, Action, https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513, 999, English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean, Counter-Strike, 5199, 10,000,000 .. 20,000,000, List(true, true, true), 201215, 999, Valve, 2000/11/1, 0, Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role., List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192), game, )",10
"List(1000000, List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud), 0, IndigoBlue Game Studio, 0, Action, Adventure, Indie, https://cdn.akamai.steamstatic.com/steam/apps/1000000/header.jpg?t=1655723048, 999, English, Korean, Simplified Chinese, ASCENXION, 5, 0 .. 20,000, List(false, false, true), 27, 999, PsychoFlux Entertainment, 2021/05/14, 0, ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, 

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

Convert the "data.tags" struct to a key/value map so that Delta can save it as an optimized Spark table with history and fast reads.

In [0]:
df_raw = df_raw.withColumn(
    "data",
    col("data").withField(
        "tags",
        from_json(to_json(col("data.tags")), MapType(StringType(), LongType()))
    )
)

(df_raw.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("steam_bronze"))

Bronze level, raw data ingestion (Steam JSON file)

In [0]:
df_bronze = spark.table("steam_bronze")

df_bronze.printSchema()
display(df_bronze.limit(3))

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

data,id
"List(10, List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP), 13990, Valve, 0, Action, https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513, 999, English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean, Counter-Strike, 5199, 10,000,000 .. 20,000,000, List(true, true, true), 201215, 999, Valve, 2000/11/1, 0, Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role., Map(1980s -> 266, PvP -> 881, Team-Based -> 1864, 1990's -> 1191, Old School -> 769, Competitive -> 1607, Score Attack -> 289, FPS -> 4831, Tactical -> 1344, Strategy -> 614, Nostalgia -> 131, Assassin -> 227, Multiplayer -> 3392, Shooter -> 3353, First-Person -> 1707, Survival -> 304, Military -> 632, Classic -> 2784, e-sports -> 1192, Action -> 5426), game, )",10
"List(1000000, List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud), 0, IndigoBlue Game Studio, 0, Action, Adventure, Indie, https://cdn.akamai.steamstatic.com/steam/apps/1000000/header.jpg?t=1655723048, 999, English, Korean, Simplified Chinese, ASCENXION, 5, 0 .. 20,000, List(false, false, true), 27, 999, PsychoFlux Entertainment, 2021/05/14, 0, ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, to uncover the truth of this world., Map(2D -> 159, Indie -> 69, Robots -> 100, Bullet Hell -> 179, Twin Stick Shooter -> 170, Side Scroller -> 175, Minimalist -> 136, Great Soundtrack -> 38, GameMaker -> 51, Controller -> 148, Colorful -> 124, Singleplayer -> 71, Abstract -> 111, Shooter -> 159, Difficult -> 161, Shoot 'Em Up -> 186, Metroidvania -> 181, Adventure -> 73, Atmospheric -> 88, Action -> 138), game, )",1000000
"List(1000010, List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards), 99, NEXT Studios, 70, Adventure, Indie, RPG, Strategy, https://cdn.akamai.steamstatic.com/steam/apps/1000010/header.jpg?t=1655724189, 1999, Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil, Crown Trick, 646, 200,000 .. 500,000, List(false, false, true), 4032, 599, Team17, NEXT Studios, 2020/10/16, 0, Enter a labyrinth that moves as you move, where mastering the elements is key to defeating enemies and uncovering the mysteries of this underground world. With a new experience awaiting every time you enter the dungeon, let the power bestowed by the crown guide you in this challenging adventure!, Map(2D -> 205, Rogue-lite -> 226, Rogue-like -> 268, Magic -> 171, Replay Value -> 192, Fantasy -> 179, Dungeon Crawler -> 225, Perma Death -> 231, Tactical -> 184, Turn-Based -> 222, Strategy -> 225, Top-Down -> 178, RPG -> 237, Singleplayer -> 211, Difficult -> 189, Turn-Based Combat -> 254, Female Protagonist -> 217, Adventure -> 225, Action-Adventure -> 179, Turn-Based Strategy -> 216), game, )",1000010


Silver level, clean and prepare data for analysis (useful columns, fix data types)

In [0]:
df_silver = df_bronze.select(
  col("data.appid").alias("appid"),
  col("data.name").alias("name"),
  col("data.publisher").alias("publisher"),
  col("data.developer").alias("developer"),
  col("data.genre").alias("genre"),
  col("data.price").alias("price"),
  col("data.discount").alias("discount"),
  col("data.initialprice").alias("initialprice"),
  col("data.release_date").alias("release_date"),
  col("data.required_age").alias("required_age"),
  col("data.positive").alias("positive"),
  col("data.negative").alias("negative"),
  col("data.owners").alias("owners"),
  col("data.platforms.windows").alias("windows"),
  col("data.platforms.mac").alias("mac"),
  col("data.platforms.linux").alias("linux"),
  col("data.languages").alias("languages"),
  col("data.categories").alias("categories"),
  col("data.short_description").alias("short_description"),
  col("data.tags").alias("tags")
)

display(df_silver.limit(5))

appid,name,publisher,developer,genre,price,discount,initialprice,release_date,required_age,positive,negative,owners,windows,mac,linux,languages,categories,short_description,tags
10,Counter-Strike,Valve,Valve,Action,999,0,999,2000/11/1,0,201215,5199,"10,000,000 .. 20,000,000",true,true,true,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean","List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"Map(1980s -> 266, PvP -> 881, Team-Based -> 1864, 1990's -> 1191, Old School -> 769, Competitive -> 1607, Score Attack -> 289, FPS -> 4831, Tactical -> 1344, Strategy -> 614, Nostalgia -> 131, Assassin -> 227, Multiplayer -> 3392, Shooter -> 3353, First-Person -> 1707, Survival -> 304, Military -> 632, Classic -> 2784, e-sports -> 1192, Action -> 5426)"
1000000,ASCENXION,PsychoFlux Entertainment,IndigoBlue Game Studio,"Action, Adventure, Indie",999,0,999,2021/05/14,0,27,5,"0 .. 20,000",true,false,false,"English, Korean, Simplified Chinese","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)","ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, to uncover the truth of this world.","Map(2D -> 159, Indie -> 69, Robots -> 100, Bullet Hell -> 179, Twin Stick Shooter -> 170, Side Scroller -> 175, Minimalist -> 136, Great Soundtrack -> 38, GameMaker -> 51, Controller -> 148, Colorful -> 124, Singleplayer -> 71, Abstract -> 111, Shooter -> 159, Difficult -> 161, Shoot 'Em Up -> 186, Metroidvania -> 181, Adventure -> 73, Atmospheric -> 88, Action -> 138)"
1000010,Crown Trick,"Team17, NEXT Studios",NEXT Studios,"Adventure, Indie, RPG, Strategy",599,70,1999,2020/10/16,0,4032,646,"200,000 .. 500,000",true,false,false,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)","Enter a labyrinth that moves as you move, where mastering the elements is key to defeating enemies and uncovering the mysteries of this underground world. With a new experience awaiting every time you enter the dungeon, let the power bestowed by the crown guide you in this challenging adventure!","Map(2D -> 205, Rogue-lite -> 226, Rogue-like -> 268, Magic -> 171, Replay Value -> 192, Fantasy -> 179, Dungeon Crawler -> 225, Perma Death -> 231, Tactical -> 184, Turn-Based -> 222, Strategy -> 225, Top-Down -> 178, RPG -> 237, Singleplayer -> 211, Difficult -> 189, Turn-Based Combat -> 254, Female Protagonist -> 217, Adventure -> 225, Action-Adventure -> 179, Turn-Based Strategy -> 216)"
1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,Vertigo Gaming Inc.,"Action, Indie, Simulation, Strategy",1999,0,1999,2020/10/14,0,1575,115,"100,000 .. 200,000",true,true,false,English,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)","Cook, serve and manage your food truck as you dish out hundreds of different foods across war-torn America in this massive sequel to the million-selling series!","Map(Family Friendly -> 175, Co-op -> 175, 2D -> 187, Simulation -> 182, Comedy -> 176, Local Multiplayer -> 158, Co-op Campaign -> 123, Arcade -> 200, Strategy -> 190, Management -> 213, Multiplayer -> 157, Funny -> 184, Single

In [0]:
df_silver.printSchema()
df_silver.dtypes

root
 |-- appid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- price: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- positive: long (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- windows: boolean (nullable = true)
 |-- mac: boolean (nullable = true)
 |-- linux: boolean (nullable = true)
 |-- languages: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- short_description: string (nullable = true)
 |-- tags: map (nullable = true)
 |    |-- key: string
 |    |-- value: long (valueContainsNull = true)

Out[6]: [('appid', 'bigint'),
 ('name', 'string'),
 ('publisher', 'string'),
 ('deve

Convert columns types

In [0]:
df_silver = (
    df_silver
    .withColumn("price", col("price").cast("float"))
    .withColumn("initialprice", col("initialprice").cast("float"))
    .withColumn("discount", col("discount").cast("float"))
    .withColumn("required_age", col("required_age").cast("int"))
    .withColumn("release_date", to_date(col("release_date"), "yyyy/M/d"))
    # create new columns (cents to euros)
    .withColumn("price_eur", round(col("price") / 100, 2))
    .withColumn("initialprice_eur", round(col("initialprice") / 100, 2))
)

df_silver.printSchema()
display(df_silver.limit(5))

root
 |-- appid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- price: float (nullable = true)
 |-- discount: float (nullable = true)
 |-- initialprice: float (nullable = true)
 |-- release_date: date (nullable = true)
 |-- required_age: integer (nullable = true)
 |-- positive: long (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- windows: boolean (nullable = true)
 |-- mac: boolean (nullable = true)
 |-- linux: boolean (nullable = true)
 |-- languages: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- short_description: string (nullable = true)
 |-- tags: map (nullable = true)
 |    |-- key: string
 |    |-- value: long (valueContainsNull = true)
 |-- price_eur: double (nullable = true)
 |-- initialprice_eur: double (nullable = true)


appid,name,publisher,developer,genre,price,discount,initialprice,release_date,required_age,positive,negative,owners,windows,mac,linux,languages,categories,short_description,tags,price_eur,initialprice_eur
10,Counter-Strike,Valve,Valve,Action,999.0,0.0,999.0,2000-11-01,0,201215,5199,"10,000,000 .. 20,000,000",true,true,true,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean","List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"Map(1980s -> 266, PvP -> 881, Team-Based -> 1864, 1990's -> 1191, Old School -> 769, Competitive -> 1607, Score Attack -> 289, FPS -> 4831, Tactical -> 1344, Strategy -> 614, Nostalgia -> 131, Assassin -> 227, Multiplayer -> 3392, Shooter -> 3353, First-Person -> 1707, Survival -> 304, Military -> 632, Classic -> 2784, e-sports -> 1192, Action -> 5426)",9.99,9.99
1000000,ASCENXION,PsychoFlux Entertainment,IndigoBlue Game Studio,"Action, Adventure, Indie",999.0,0.0,999.0,2021-05-14,0,27,5,"0 .. 20,000",true,false,false,"English, Korean, Simplified Chinese","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)","ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, to uncover the truth of this world.","Map(2D -> 159, Indie -> 69, Robots -> 100, Bullet Hell -> 179, Twin Stick Shooter -> 170, Side Scroller -> 175, Minimalist -> 136, Great Soundtrack -> 38, GameMaker -> 51, Controller -> 148, Colorful -> 124, Singleplayer -> 71, Abstract -> 111, Shooter -> 159, Difficult -> 161, Shoot 'Em Up -> 186, Metroidvania -> 181, Adventure -> 73, Atmospheric -> 88, Action -> 138)",9.99,9.99
1000010,Crown Trick,"Team17, NEXT Studios",NEXT Studios,"Adventure, Indie, RPG, Strategy",599.0,70.0,1999.0,2020-10-16,0,4032,646,"200,000 .. 500,000",true,false,false,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)","Enter a labyrinth that moves as you move, where mastering the elements is key to defeating enemies and uncovering the mysteries of this underground world. With a new experience awaiting every time you enter the dungeon, let the power bestowed by the crown guide you in this challenging adventure!","Map(2D -> 205, Rogue-lite -> 226, Rogue-like -> 268, Magic -> 171, Replay Value -> 192, Fantasy -> 179, Dungeon Crawler -> 225, Perma Death -> 231, Tactical -> 184, Turn-Based -> 222, Strategy -> 225, Top-Down -> 178, RPG -> 237, Singleplayer -> 211, Difficult -> 189, Turn-Based Combat -> 254, Female Protagonist -> 217, Adventure -> 225, Action-Adventure -> 179, Turn-Based Strategy -> 216)",5.99,19.99
1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,Vertigo Gaming Inc.,"Action, Indie, Simulation, Strategy",1999.0,0.0,1999.0,2020-10-14,0,1575,115,"100,000 .. 200,000",true,true,false,English,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)","Cook, serve and manage your food truck as you dish out hundreds of different foods across war-torn America in this massive sequel to the million-selling series!","Map(Family Friendly -> 175, Co-op -> 175, 2D -> 187, Simulation -> 182, Comedy -> 176, Local Multiplayer -> 158, Co-op Campaign -> 123, Arcade -

In [0]:
df_silver.columns

Out[8]: ['appid',
 'name',
 'publisher',
 'developer',
 'genre',
 'price',
 'discount',
 'initialprice',
 'release_date',
 'required_age',
 'positive',
 'negative',
 'owners',
 'windows',
 'mac',
 'linux',
 'languages',
 'categories',
 'short_description',
 'tags',
 'price_eur',
 'initialprice_eur']

In [0]:
df_silver = df_silver.select(
    "appid",
    "name",
    "publisher",
    "developer",
    "genre",
    "price",
    "price_eur",
    "discount",
    "initialprice",
    "initialprice_eur",
    "release_date",
    "required_age",
    "positive",
    "negative",
    "owners",
    "windows",
    "mac",
    "linux",
    "languages",
    "categories",
    "short_description",
    "tags"
)

display(df_silver.limit(5))


appid,name,publisher,developer,genre,price,price_eur,discount,initialprice,initialprice_eur,release_date,required_age,positive,negative,owners,windows,mac,linux,languages,categories,short_description,tags
10,Counter-Strike,Valve,Valve,Action,999.0,9.99,0.0,999.0,9.99,2000-11-01,0,201215,5199,"10,000,000 .. 20,000,000",true,true,true,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean","List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"Map(1980s -> 266, PvP -> 881, Team-Based -> 1864, 1990's -> 1191, Old School -> 769, Competitive -> 1607, Score Attack -> 289, FPS -> 4831, Tactical -> 1344, Strategy -> 614, Nostalgia -> 131, Assassin -> 227, Multiplayer -> 3392, Shooter -> 3353, First-Person -> 1707, Survival -> 304, Military -> 632, Classic -> 2784, e-sports -> 1192, Action -> 5426)"
1000000,ASCENXION,PsychoFlux Entertainment,IndigoBlue Game Studio,"Action, Adventure, Indie",999.0,9.99,0.0,999.0,9.99,2021-05-14,0,27,5,"0 .. 20,000",true,false,false,"English, Korean, Simplified Chinese","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)","ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, to uncover the truth of this world.","Map(2D -> 159, Indie -> 69, Robots -> 100, Bullet Hell -> 179, Twin Stick Shooter -> 170, Side Scroller -> 175, Minimalist -> 136, Great Soundtrack -> 38, GameMaker -> 51, Controller -> 148, Colorful -> 124, Singleplayer -> 71, Abstract -> 111, Shooter -> 159, Difficult -> 161, Shoot 'Em Up -> 186, Metroidvania -> 181, Adventure -> 73, Atmospheric -> 88, Action -> 138)"
1000010,Crown Trick,"Team17, NEXT Studios",NEXT Studios,"Adventure, Indie, RPG, Strategy",599.0,5.99,70.0,1999.0,19.99,2020-10-16,0,4032,646,"200,000 .. 500,000",true,false,false,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)","Enter a labyrinth that moves as you move, where mastering the elements is key to defeating enemies and uncovering the mysteries of this underground world. With a new experience awaiting every time you enter the dungeon, let the power bestowed by the crown guide you in this challenging adventure!","Map(2D -> 205, Rogue-lite -> 226, Rogue-like -> 268, Magic -> 171, Replay Value -> 192, Fantasy -> 179, Dungeon Crawler -> 225, Perma Death -> 231, Tactical -> 184, Turn-Based -> 222, Strategy -> 225, Top-Down -> 178, RPG -> 237, Singleplayer -> 211, Difficult -> 189, Turn-Based Combat -> 254, Female Protagonist -> 217, Adventure -> 225, Action-Adventure -> 179, Turn-Based Strategy -> 216)"
1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,Vertigo Gaming Inc.,"Action, Indie, Simulation, Strategy",1999.0,19.99,0.0,1999.0,19.99,2020-10-14,0,1575,115,"100,000 .. 200,000",true,true,false,English,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)","Cook, serve and manage your food truck as you dish out hundreds of different foods across war-torn America in this massive sequel to the million-selling series!","Map(Family Friendly -> 175, Co-op -> 175, 2D -> 187, Simulation -> 182, Comedy -> 176, Local Multiplayer -> 158, Co-op Campaign -> 1

In [0]:
(df_silver.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("steam_silver"))

Gold level, create an enriched dataset ready for analysis  
(add key metrics total ratings, positive review percentage, and release year)

In [0]:
df_silver = spark.table("steam_silver")
display(df_silver.limit(5))

appid,name,publisher,developer,genre,price,price_eur,discount,initialprice,initialprice_eur,release_date,required_age,positive,negative,owners,windows,mac,linux,languages,categories,short_description,tags
10,Counter-Strike,Valve,Valve,Action,999.0,9.99,0.0,999.0,9.99,2000-11-01,0,201215,5199,"10,000,000 .. 20,000,000",true,true,true,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean","List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"Map(1980s -> 266, PvP -> 881, Team-Based -> 1864, 1990's -> 1191, Old School -> 769, Competitive -> 1607, Score Attack -> 289, FPS -> 4831, Tactical -> 1344, Strategy -> 614, Nostalgia -> 131, Assassin -> 227, Multiplayer -> 3392, Shooter -> 3353, First-Person -> 1707, Survival -> 304, Military -> 632, Classic -> 2784, e-sports -> 1192, Action -> 5426)"
1000000,ASCENXION,PsychoFlux Entertainment,IndigoBlue Game Studio,"Action, Adventure, Indie",999.0,9.99,0.0,999.0,9.99,2021-05-14,0,27,5,"0 .. 20,000",true,false,false,"English, Korean, Simplified Chinese","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)","ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, to uncover the truth of this world.","Map(2D -> 159, Indie -> 69, Robots -> 100, Bullet Hell -> 179, Twin Stick Shooter -> 170, Side Scroller -> 175, Minimalist -> 136, Great Soundtrack -> 38, GameMaker -> 51, Controller -> 148, Colorful -> 124, Singleplayer -> 71, Abstract -> 111, Shooter -> 159, Difficult -> 161, Shoot 'Em Up -> 186, Metroidvania -> 181, Adventure -> 73, Atmospheric -> 88, Action -> 138)"
1000010,Crown Trick,"Team17, NEXT Studios",NEXT Studios,"Adventure, Indie, RPG, Strategy",599.0,5.99,70.0,1999.0,19.99,2020-10-16,0,4032,646,"200,000 .. 500,000",true,false,false,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)","Enter a labyrinth that moves as you move, where mastering the elements is key to defeating enemies and uncovering the mysteries of this underground world. With a new experience awaiting every time you enter the dungeon, let the power bestowed by the crown guide you in this challenging adventure!","Map(2D -> 205, Rogue-lite -> 226, Rogue-like -> 268, Magic -> 171, Replay Value -> 192, Fantasy -> 179, Dungeon Crawler -> 225, Perma Death -> 231, Tactical -> 184, Turn-Based -> 222, Strategy -> 225, Top-Down -> 178, RPG -> 237, Singleplayer -> 211, Difficult -> 189, Turn-Based Combat -> 254, Female Protagonist -> 217, Adventure -> 225, Action-Adventure -> 179, Turn-Based Strategy -> 216)"
1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,Vertigo Gaming Inc.,"Action, Indie, Simulation, Strategy",1999.0,19.99,0.0,1999.0,19.99,2020-10-14,0,1575,115,"100,000 .. 200,000",true,true,false,English,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)","Cook, serve and manage your food truck as you dish out hundreds of different foods across war-torn America in this massive sequel to the million-selling series!","Map(Family Friendly -> 175, Co-op -> 175, 2D -> 187, Simulation -> 182, Comedy -> 176, Local Multiplayer -> 158, Co-op Campaign -> 1

In [0]:
df_gold = (
  df_silver
  .withColumn("total_ratings", (col("positive") + col("negative")).cast("long"))
  .withColumn("per_positive_share", round((col("positive")/(col("positive") + col("negative"))) * 100, 1))
  .withColumn("release_year", year(col("release_date")))
)

df_gold.printSchema()
display(df_gold.limit(5))


root
 |-- appid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- price: float (nullable = true)
 |-- price_eur: double (nullable = true)
 |-- discount: float (nullable = true)
 |-- initialprice: float (nullable = true)
 |-- initialprice_eur: double (nullable = true)
 |-- release_date: date (nullable = true)
 |-- required_age: integer (nullable = true)
 |-- positive: long (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- windows: boolean (nullable = true)
 |-- mac: boolean (nullable = true)
 |-- linux: boolean (nullable = true)
 |-- languages: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- short_description: string (nullable = true)
 |-- tags: map (nullable = true)
 |    |-- key: string
 |    |-- value: long (valueContainsNull = true)


appid,name,publisher,developer,genre,price,price_eur,discount,initialprice,initialprice_eur,release_date,required_age,positive,negative,owners,windows,mac,linux,languages,categories,short_description,tags,total_ratings,per_positive_share,release_year
10,Counter-Strike,Valve,Valve,Action,999.0,9.99,0.0,999.0,9.99,2000-11-01,0,201215,5199,"10,000,000 .. 20,000,000",true,true,true,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean","List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"Map(1980s -> 266, PvP -> 881, Team-Based -> 1864, 1990's -> 1191, Old School -> 769, Competitive -> 1607, Score Attack -> 289, FPS -> 4831, Tactical -> 1344, Strategy -> 614, Nostalgia -> 131, Assassin -> 227, Multiplayer -> 3392, Shooter -> 3353, First-Person -> 1707, Survival -> 304, Military -> 632, Classic -> 2784, e-sports -> 1192, Action -> 5426)",206414,97.5,2000
1000000,ASCENXION,PsychoFlux Entertainment,IndigoBlue Game Studio,"Action, Adventure, Indie",999.0,9.99,0.0,999.0,9.99,2021-05-14,0,27,5,"0 .. 20,000",true,false,false,"English, Korean, Simplified Chinese","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)","ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, to uncover the truth of this world.","Map(2D -> 159, Indie -> 69, Robots -> 100, Bullet Hell -> 179, Twin Stick Shooter -> 170, Side Scroller -> 175, Minimalist -> 136, Great Soundtrack -> 38, GameMaker -> 51, Controller -> 148, Colorful -> 124, Singleplayer -> 71, Abstract -> 111, Shooter -> 159, Difficult -> 161, Shoot 'Em Up -> 186, Metroidvania -> 181, Adventure -> 73, Atmospheric -> 88, Action -> 138)",32,84.4,2021
1000010,Crown Trick,"Team17, NEXT Studios",NEXT Studios,"Adventure, Indie, RPG, Strategy",599.0,5.99,70.0,1999.0,19.99,2020-10-16,0,4032,646,"200,000 .. 500,000",true,false,false,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)","Enter a labyrinth that moves as you move, where mastering the elements is key to defeating enemies and uncovering the mysteries of this underground world. With a new experience awaiting every time you enter the dungeon, let the power bestowed by the crown guide you in this challenging adventure!","Map(2D -> 205, Rogue-lite -> 226, Rogue-like -> 268, Magic -> 171, Replay Value -> 192, Fantasy -> 179, Dungeon Crawler -> 225, Perma Death -> 231, Tactical -> 184, Turn-Based -> 222, Strategy -> 225, Top-Down -> 178, RPG -> 237, Singleplayer -> 211, Difficult -> 189, Turn-Based Combat -> 254, Female Protagonist -> 217, Adventure -> 225, Action-Adventure -> 179, Turn-Based Strategy -> 216)",4678,86.2,2020
1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,Vertigo Gaming Inc.,"Action, Indie, Simulation, Strategy",1999.0,19.99,0.0,1999.0,19.99,2020-10-14,0,1575,115,"100,000 .. 200,000",true,true,false,English,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)","Cook, serve and manage your food truck as you dish out hundreds of different foods across war-torn America in this massive sequel to the million-selling series!","Map(Family Friendly -> 175, Co-op -> 175,

In [0]:
df_gold.columns

Out[13]: ['appid',
 'name',
 'publisher',
 'developer',
 'genre',
 'price',
 'price_eur',
 'discount',
 'initialprice',
 'initialprice_eur',
 'release_date',
 'required_age',
 'positive',
 'negative',
 'owners',
 'windows',
 'mac',
 'linux',
 'languages',
 'categories',
 'short_description',
 'tags',
 'total_ratings',
 'per_positive_share',
 'release_year']

In [0]:
df_gold = df_gold.select(
    "appid",
    "name",
    "publisher",
    "developer",
    "genre",
    "price",
    "price_eur",
    "discount",
    "initialprice",
    "initialprice_eur",
    "release_date",
    "release_year",
    "required_age",
    "positive",
    "negative",
    "total_ratings",   
    "per_positive_share",
    "owners",
    "windows",
    "mac",
    "linux",
    "languages",
    "categories",
    "short_description",
    "tags",
)

display(df_gold.limit(5))

appid,name,publisher,developer,genre,price,price_eur,discount,initialprice,initialprice_eur,release_date,release_year,required_age,positive,negative,total_ratings,per_positive_share,owners,windows,mac,linux,languages,categories,short_description,tags
10,Counter-Strike,Valve,Valve,Action,999.0,9.99,0.0,999.0,9.99,2000-11-01,2000,0,201215,5199,206414,97.5,"10,000,000 .. 20,000,000",true,true,true,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean","List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"Map(1980s -> 266, PvP -> 881, Team-Based -> 1864, 1990's -> 1191, Old School -> 769, Competitive -> 1607, Score Attack -> 289, FPS -> 4831, Tactical -> 1344, Strategy -> 614, Nostalgia -> 131, Assassin -> 227, Multiplayer -> 3392, Shooter -> 3353, First-Person -> 1707, Survival -> 304, Military -> 632, Classic -> 2784, e-sports -> 1192, Action -> 5426)"
1000000,ASCENXION,PsychoFlux Entertainment,IndigoBlue Game Studio,"Action, Adventure, Indie",999.0,9.99,0.0,999.0,9.99,2021-05-14,2021,0,27,5,32,84.4,"0 .. 20,000",true,false,false,"English, Korean, Simplified Chinese","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)","ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, to uncover the truth of this world.","Map(2D -> 159, Indie -> 69, Robots -> 100, Bullet Hell -> 179, Twin Stick Shooter -> 170, Side Scroller -> 175, Minimalist -> 136, Great Soundtrack -> 38, GameMaker -> 51, Controller -> 148, Colorful -> 124, Singleplayer -> 71, Abstract -> 111, Shooter -> 159, Difficult -> 161, Shoot 'Em Up -> 186, Metroidvania -> 181, Adventure -> 73, Atmospheric -> 88, Action -> 138)"
1000010,Crown Trick,"Team17, NEXT Studios",NEXT Studios,"Adventure, Indie, RPG, Strategy",599.0,5.99,70.0,1999.0,19.99,2020-10-16,2020,0,4032,646,4678,86.2,"200,000 .. 500,000",true,false,false,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)","Enter a labyrinth that moves as you move, where mastering the elements is key to defeating enemies and uncovering the mysteries of this underground world. With a new experience awaiting every time you enter the dungeon, let the power bestowed by the crown guide you in this challenging adventure!","Map(2D -> 205, Rogue-lite -> 226, Rogue-like -> 268, Magic -> 171, Replay Value -> 192, Fantasy -> 179, Dungeon Crawler -> 225, Perma Death -> 231, Tactical -> 184, Turn-Based -> 222, Strategy -> 225, Top-Down -> 178, RPG -> 237, Singleplayer -> 211, Difficult -> 189, Turn-Based Combat -> 254, Female Protagonist -> 217, Adventure -> 225, Action-Adventure -> 179, Turn-Based Strategy -> 216)"
1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,Vertigo Gaming Inc.,"Action, Indie, Simulation, Strategy",1999.0,19.99,0.0,1999.0,19.99,2020-10-14,2020,0,1575,115,1690,93.2,"100,000 .. 200,000",true,true,false,English,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)","Cook, serve and manage your food truck as you dish out hundreds of different foods across war-torn America in this massive sequel to the million-selling series!","Map(Family Friendly -> 175

In [0]:
(df_gold.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("steam_gold"))

In [0]:
df_gold = spark.table("steam_gold")
display(df_gold.limit(5))

appid,name,publisher,developer,genre,price,price_eur,discount,initialprice,initialprice_eur,release_date,release_year,required_age,positive,negative,total_ratings,per_positive_share,owners,windows,mac,linux,languages,categories,short_description,tags
10,Counter-Strike,Valve,Valve,Action,999.0,9.99,0.0,999.0,9.99,2000-11-01,2000,0,201215,5199,206414,97.5,"10,000,000 .. 20,000,000",true,true,true,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean","List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"Map(1980s -> 266, PvP -> 881, Team-Based -> 1864, 1990's -> 1191, Old School -> 769, Competitive -> 1607, Score Attack -> 289, FPS -> 4831, Tactical -> 1344, Strategy -> 614, Nostalgia -> 131, Assassin -> 227, Multiplayer -> 3392, Shooter -> 3353, First-Person -> 1707, Survival -> 304, Military -> 632, Classic -> 2784, e-sports -> 1192, Action -> 5426)"
1000000,ASCENXION,PsychoFlux Entertainment,IndigoBlue Game Studio,"Action, Adventure, Indie",999.0,9.99,0.0,999.0,9.99,2021-05-14,2021,0,27,5,32,84.4,"0 .. 20,000",true,false,false,"English, Korean, Simplified Chinese","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)","ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, to uncover the truth of this world.","Map(2D -> 159, Indie -> 69, Robots -> 100, Bullet Hell -> 179, Twin Stick Shooter -> 170, Side Scroller -> 175, Minimalist -> 136, Great Soundtrack -> 38, GameMaker -> 51, Controller -> 148, Colorful -> 124, Singleplayer -> 71, Abstract -> 111, Shooter -> 159, Difficult -> 161, Shoot 'Em Up -> 186, Metroidvania -> 181, Adventure -> 73, Atmospheric -> 88, Action -> 138)"
1000010,Crown Trick,"Team17, NEXT Studios",NEXT Studios,"Adventure, Indie, RPG, Strategy",599.0,5.99,70.0,1999.0,19.99,2020-10-16,2020,0,4032,646,4678,86.2,"200,000 .. 500,000",true,false,false,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)","Enter a labyrinth that moves as you move, where mastering the elements is key to defeating enemies and uncovering the mysteries of this underground world. With a new experience awaiting every time you enter the dungeon, let the power bestowed by the crown guide you in this challenging adventure!","Map(2D -> 205, Rogue-lite -> 226, Rogue-like -> 268, Magic -> 171, Replay Value -> 192, Fantasy -> 179, Dungeon Crawler -> 225, Perma Death -> 231, Tactical -> 184, Turn-Based -> 222, Strategy -> 225, Top-Down -> 178, RPG -> 237, Singleplayer -> 211, Difficult -> 189, Turn-Based Combat -> 254, Female Protagonist -> 217, Adventure -> 225, Action-Adventure -> 179, Turn-Based Strategy -> 216)"
1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,Vertigo Gaming Inc.,"Action, Indie, Simulation, Strategy",1999.0,19.99,0.0,1999.0,19.99,2020-10-14,2020,0,1575,115,1690,93.2,"100,000 .. 200,000",true,true,false,English,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)","Cook, serve and manage your food truck as you dish out hundreds of different foods across war-torn America in this massive sequel to the million-selling series!","Map(Family Friendly -> 175

### Analysis at the "macro" level
Which publisher has released the most games on Steam?

In [0]:
df_publisher= (
    df_gold
    .groupBy("publisher")
    .count()
    .orderBy(col("count").desc())
)

display(df_publisher.limit(20))

publisher,count
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141
Choice of Games,140
Sekai Project,132
HH-Games,132
,132
Ubisoft,127


In [0]:
df_gold.select("publisher").where(col("publisher").isNull() | (col("publisher") == "")).count()

132

In [0]:
df_publisher= (
    df_gold
    .where(col("publisher").isNotNull() & (col("publisher") != ""))
    .groupBy("publisher")
    .count()
    .orderBy(col("count").desc())
)

display(df_publisher.limit(20))

publisher,count
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141
Choice of Games,140
Sekai Project,132
HH-Games,132
Ubisoft,127
Laush Studio,126


Databricks visualization. Run in Databricks to view.

Big Fish Games is by far the publisher that has released the highest number of games on Steam.  
8floor and SEGA follow just behind.

What are the best rated games?

In [0]:
df_games = (
  df_gold
  .select("name", "publisher", "total_ratings", "per_positive_share")
  .where(col("total_ratings") > 1000)
  .orderBy(col("per_positive_share").desc())
)

display(df_games.limit(20))

name,publisher,total_ratings,per_positive_share
Aventura Copilului Albastru și Urât,Codrin Bradea,2217,99.4
A Short Hike,adamgryu,11732,99.3
Patrick's Parabox,Patrick Traynor,1500,99.3
Aseprite,Igara Studio,11903,99.3
Senren＊Banka,"HIKARI FIELD, NekoNyan Ltd.",10677,99.2
Monster Prom 3: Monster Roadtrip,Beautiful Glitch,1060,99.2
CULTIC,3D Realms,2037,99.2
Ib,PLAYISM,1829,99.2
星空列车与白的旅行,"inc.ZOFE, Syawase Works China",1563,99.1
TOEM,Something We Made,1539,99.0


Aventura Copilului Albastru și Urât” ranks first with a 99.4 % positive rate but only around 2 000 reviews, whereas “A Short Hike” has a similar score with over 11 000 reviews, making it a more reliable rating. Interestingly, none of these top-rated games are published by the top 20 most prolific publishers

Are there years with more releases? Were there more or fewer game releases during the Covid, for example?

In [0]:
df_years = (
    df_gold
    .groupBy("release_year")
    .count()
    .orderBy("release_year")
)

display(df_years)

release_year,count
null,222
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6


In [0]:
df_years = (
    df_gold
    .where(col("release_year").isNotNull())
    .groupBy("release_year")
    .count()
    .orderBy("release_year")
)

display(df_years)

release_year,count
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61


Databricks visualization. Run in Databricks to view.

Game releases on Steam grew sharply from 2014 onward, peaking between 2020 and 2021 with over 8 000 titles per year.  
The Covid period actually saw record-high releases.

How are the prizes distributed? Are there many games with a discount?

In [0]:
display(
    df_gold
    .select("price_eur")
    .where((col("price_eur").isNotNull()))
)

price_eur
9.99
9.99
5.99
19.99
1.99
7.99
12.99
0.0
2.99
13.99


Databricks visualization. Run in Databricks to view.

In [0]:
df_gold.select(max("price_eur")).show()

+--------------+
|max(price_eur)|
+--------------+
|         999.0|
+--------------+



In [0]:
df_gold.where(col("price_eur") > 100).count()

37

In [0]:
display(
    df_gold
    .select("price_eur")
    .where((col("price_eur").isNotNull()) & (col("price_eur") <= 100))
)

price_eur
9.99
9.99
5.99
19.99
1.99
7.99
12.99
0.0
2.99
13.99


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    df_gold
    .select("name", "publisher", "price_eur")
    .where(col("price_eur") > 100)
)

name,publisher,price_eur
Lgnorant girl doll,wandwad,199.99
VR-CPR Personal Edition,"Studio Evil, IRC Edu",119.99
眼睛（眼球）结构研究,上海皋城软件有限公司,199.99
Run Thief,H.G.G.,199.99
CyberLink AudioDirector 10 Ultra,"Deep Silver, CyberLink",129.99
MixPad,NCH Software,149.99
Ascent Free-Roaming VR Experience,Fury Games,999.0
Spot Sample Witness Simulator,"Kelley Integrity Safety Solutions, LLC, Endless Simulations",199.99
VBOX Sim,Racelogic,129.99
VR Long March,RK,199.99


Databricks visualization. Run in Databricks to view.

Most games on Steam are priced under €20, showing a strong focus on affordable titles. A few entries (37) appear above 100€, with one reaching 999€, maybe a mistake.

In [0]:
df_discount = (
  df_gold
  .withColumn("has_discount", when(col("discount").isNotNull() & (col("discount") > 0), "Yes").otherwise("No"))
  .groupBy("has_discount")
  .count()
)

display(df_discount)

has_discount,count
No,53173
Yes,2518


Databricks visualization. Run in Databricks to view.

Only about 4.5 % of games currently show a discount

What are the most represented languages?

In [0]:
display(df_gold.select("languages").limit(10))

languages
"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean"
"English, Korean, Simplified Chinese"
"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil"
English
Simplified Chinese
"Simplified Chinese, English, Traditional Chinese, Japanese, Korean"
"Japanese, Simplified Chinese, Traditional Chinese"
"English, Simplified Chinese, Traditional Chinese"
English
"English, Simplified Chinese, Traditional Chinese"


In [0]:
df_lang = (
    df_gold
    .where(col("languages").isNotNull())
    .withColumn("language", explode(split(col("languages"), ",")))
    .withColumn("language", trim(lower(col("language"))))
    .groupBy("language")
    .count()
    .orderBy(col("count").desc())
)

display(df_lang.limit(20))

language,count
english,55116
german,14019
french,13426
russian,12922
simplified chinese,12782
spanish - spain,12233
japanese,10368
italian,9304
portuguese - brazil,6750
korean,6600


Databricks visualization. Run in Databricks to view.

English is by far the most represented language on Steam, followed by German, French, and Russian.

Are there many games prohibited for children under 16/18?

In [0]:
display(df_gold.limit(5))

appid,name,publisher,developer,genre,price,price_eur,discount,initialprice,initialprice_eur,release_date,release_year,required_age,positive,negative,total_ratings,per_positive_share,owners,windows,mac,linux,languages,categories,short_description,tags
10,Counter-Strike,Valve,Valve,Action,999.0,9.99,0.0,999.0,9.99,2000-11-01,2000,0,201215,5199,206414,97.5,"10,000,000 .. 20,000,000",true,true,true,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean","List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"Map(1980s -> 266, PvP -> 881, Team-Based -> 1864, 1990's -> 1191, Old School -> 769, Competitive -> 1607, Score Attack -> 289, FPS -> 4831, Tactical -> 1344, Strategy -> 614, Nostalgia -> 131, Assassin -> 227, Multiplayer -> 3392, Shooter -> 3353, First-Person -> 1707, Survival -> 304, Military -> 632, Classic -> 2784, e-sports -> 1192, Action -> 5426)"
1000000,ASCENXION,PsychoFlux Entertainment,IndigoBlue Game Studio,"Action, Adventure, Indie",999.0,9.99,0.0,999.0,9.99,2021-05-14,2021,0,27,5,32,84.4,"0 .. 20,000",true,false,false,"English, Korean, Simplified Chinese","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)","ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, to uncover the truth of this world.","Map(2D -> 159, Indie -> 69, Robots -> 100, Bullet Hell -> 179, Twin Stick Shooter -> 170, Side Scroller -> 175, Minimalist -> 136, Great Soundtrack -> 38, GameMaker -> 51, Controller -> 148, Colorful -> 124, Singleplayer -> 71, Abstract -> 111, Shooter -> 159, Difficult -> 161, Shoot 'Em Up -> 186, Metroidvania -> 181, Adventure -> 73, Atmospheric -> 88, Action -> 138)"
1000010,Crown Trick,"Team17, NEXT Studios",NEXT Studios,"Adventure, Indie, RPG, Strategy",599.0,5.99,70.0,1999.0,19.99,2020-10-16,2020,0,4032,646,4678,86.2,"200,000 .. 500,000",true,false,false,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)","Enter a labyrinth that moves as you move, where mastering the elements is key to defeating enemies and uncovering the mysteries of this underground world. With a new experience awaiting every time you enter the dungeon, let the power bestowed by the crown guide you in this challenging adventure!","Map(2D -> 205, Rogue-lite -> 226, Rogue-like -> 268, Magic -> 171, Replay Value -> 192, Fantasy -> 179, Dungeon Crawler -> 225, Perma Death -> 231, Tactical -> 184, Turn-Based -> 222, Strategy -> 225, Top-Down -> 178, RPG -> 237, Singleplayer -> 211, Difficult -> 189, Turn-Based Combat -> 254, Female Protagonist -> 217, Adventure -> 225, Action-Adventure -> 179, Turn-Based Strategy -> 216)"
1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,Vertigo Gaming Inc.,"Action, Indie, Simulation, Strategy",1999.0,19.99,0.0,1999.0,19.99,2020-10-14,2020,0,1575,115,1690,93.2,"100,000 .. 200,000",true,true,false,English,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)","Cook, serve and manage your food truck as you dish out hundreds of different foods across war-torn America in this massive sequel to the million-selling series!","Map(Family Friendly -> 175

In [0]:
display(
    df_gold
    .groupBy("required_age")
    .count()
    .orderBy(col("required_age"))
)

required_age,count
null,3
0,55030
3,3
5,1
6,4
7,2
8,3
9,1
10,7
12,32


In [0]:
display(
    df_gold
    .where(col("required_age").isNotNull())
    .groupBy("required_age")
    .count()
    .orderBy(col("required_age"))
)

required_age,count
0,55030
3,3
5,1
6,4
7,2
8,3
9,1
10,7
12,32
13,26


In [0]:
df_age = (
    df_gold
    .where(col("required_age").isNotNull())
    .withColumn(
        "age_group",
        when(col("required_age") >= 16, "16+ / 18+").otherwise("All ages")
    )
    .groupBy("age_group")
    .count()
)

display(df_age)

age_group,count
16+ / 18+,305
All ages,55383


Databricks visualization. Run in Databricks to view.

Almost all games on Steam are available for all audiences, with only a very small fraction (around 0.5%) restricted to players aged 16 or 18 and above.

### Genres analysis

What are the most represented genres?

In [0]:
display(df_gold.limit(5))

appid,name,publisher,developer,genre,price,price_eur,discount,initialprice,initialprice_eur,release_date,release_year,required_age,positive,negative,total_ratings,per_positive_share,owners,windows,mac,linux,languages,categories,short_description,tags
10,Counter-Strike,Valve,Valve,Action,999.0,9.99,0.0,999.0,9.99,2000-11-01,2000,0,201215,5199,206414,97.5,"10,000,000 .. 20,000,000",true,true,true,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean","List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"Map(1980s -> 266, PvP -> 881, Team-Based -> 1864, 1990's -> 1191, Old School -> 769, Competitive -> 1607, Score Attack -> 289, FPS -> 4831, Tactical -> 1344, Strategy -> 614, Nostalgia -> 131, Assassin -> 227, Multiplayer -> 3392, Shooter -> 3353, First-Person -> 1707, Survival -> 304, Military -> 632, Classic -> 2784, e-sports -> 1192, Action -> 5426)"
1000000,ASCENXION,PsychoFlux Entertainment,IndigoBlue Game Studio,"Action, Adventure, Indie",999.0,9.99,0.0,999.0,9.99,2021-05-14,2021,0,27,5,32,84.4,"0 .. 20,000",true,false,false,"English, Korean, Simplified Chinese","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)","ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, to uncover the truth of this world.","Map(2D -> 159, Indie -> 69, Robots -> 100, Bullet Hell -> 179, Twin Stick Shooter -> 170, Side Scroller -> 175, Minimalist -> 136, Great Soundtrack -> 38, GameMaker -> 51, Controller -> 148, Colorful -> 124, Singleplayer -> 71, Abstract -> 111, Shooter -> 159, Difficult -> 161, Shoot 'Em Up -> 186, Metroidvania -> 181, Adventure -> 73, Atmospheric -> 88, Action -> 138)"
1000010,Crown Trick,"Team17, NEXT Studios",NEXT Studios,"Adventure, Indie, RPG, Strategy",599.0,5.99,70.0,1999.0,19.99,2020-10-16,2020,0,4032,646,4678,86.2,"200,000 .. 500,000",true,false,false,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)","Enter a labyrinth that moves as you move, where mastering the elements is key to defeating enemies and uncovering the mysteries of this underground world. With a new experience awaiting every time you enter the dungeon, let the power bestowed by the crown guide you in this challenging adventure!","Map(2D -> 205, Rogue-lite -> 226, Rogue-like -> 268, Magic -> 171, Replay Value -> 192, Fantasy -> 179, Dungeon Crawler -> 225, Perma Death -> 231, Tactical -> 184, Turn-Based -> 222, Strategy -> 225, Top-Down -> 178, RPG -> 237, Singleplayer -> 211, Difficult -> 189, Turn-Based Combat -> 254, Female Protagonist -> 217, Adventure -> 225, Action-Adventure -> 179, Turn-Based Strategy -> 216)"
1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,Vertigo Gaming Inc.,"Action, Indie, Simulation, Strategy",1999.0,19.99,0.0,1999.0,19.99,2020-10-14,2020,0,1575,115,1690,93.2,"100,000 .. 200,000",true,true,false,English,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)","Cook, serve and manage your food truck as you dish out hundreds of different foods across war-torn America in this massive sequel to the million-selling series!","Map(Family Friendly -> 175

In [0]:
df_genres = (
    df_gold
    .where(col("genre").isNotNull())
    .withColumn("genre_item", explode(split(col("genre"), ",")))
    .withColumn("genre_item", trim(lower(col("genre_item"))))
    .groupBy("genre_item")
    .count()
    .orderBy(col("count").desc())
)

display(df_genres.limit(20))

genre_item,count
indie,39681
action,23759
casual,22086
adventure,21431
strategy,10895
simulation,10836
rpg,9534
early access,6145
free to play,3393
sports,2666


Databricks visualization. Run in Databricks to view.

The Indie genre dominates Steam’s catalog with nearly 40 000 titles, follow by Action, Casual, and Adventure games.

Are there any genres that have a better positive/negative review ratio?

In [0]:
df_genre_full = (
    df_gold
    .where(col("genre").isNotNull())
    .withColumn("genre_item", explode(split(col("genre"), ",")))
    .withColumn("genre_item", trim(lower(col("genre_item"))))
    .select(
        "name", 
        "publisher", 
        "genre_item", 
        "per_positive_share", 
        "total_ratings",
        "price_eur"
    )
)

display(df_genre_full.limit(20))

name,publisher,genre_item,per_positive_share,total_ratings,price_eur
Counter-Strike,Valve,action,97.5,206414,9.99
ASCENXION,PsychoFlux Entertainment,action,84.4,32,9.99
ASCENXION,PsychoFlux Entertainment,adventure,84.4,32,9.99
ASCENXION,PsychoFlux Entertainment,indie,84.4,32,9.99
Crown Trick,"Team17, NEXT Studios",adventure,86.2,4678,5.99
Crown Trick,"Team17, NEXT Studios",indie,86.2,4678,5.99
Crown Trick,"Team17, NEXT Studios",rpg,86.2,4678,5.99
Crown Trick,"Team17, NEXT Studios",strategy,86.2,4678,5.99
"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,action,93.2,1690,19.99
"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,indie,93.2,1690,19.99


In [0]:
df_genre_score = (
    df_genre_full
    .where((col("genre_item").isNotNull()) & (col("genre_item") != ""))
    .groupBy("genre_item")
    .agg(
        round(avg("per_positive_share"), 1).alias("avg_positive_share"),
        count("*").alias("count_games")
    )
    .where(col("count_games") >= 100)
    .orderBy(col("avg_positive_share").desc())
)

display(df_genre_score.limit(20))

genre_item,avg_positive_share,count_games
casual,74.3,22086
indie,74.2,39681
adventure,73.9,21431
game development,73.8,159
rpg,73.1,9534
action,73.0,23759
free to play,72.2,3393
design & illustration,72.1,406
strategy,71.9,10895
early access,70.7,6145


Casual, indie, and adventure games show the highest average positive review ratios, all around 74 %.

In [0]:
display(
    df_genre_score
    .orderBy(col("avg_positive_share").asc())
    .limit(20)
)

genre_item,avg_positive_share,count_games
violent,59.1,168
massively multiplayer,62.9,1460
audio production,67.5,195
video production,68.0,247
education,68.5,317
simulation,69.4,10836
sports,69.8,2666
utilities,69.9,682
software training,70.0,164
racing,70.2,2155


Violent, massively multiplayer, and simulation games record the lowest average positive review ratios.

Do some publishers have favorite genres?

In [0]:
df_pub_genre = (
  df_genre_full
  .where((col("publisher").isNotNull()) & (col("publisher") != ""))
  .groupby("publisher", "genre_item")
  .count()
  .orderBy(col("count").desc())
)

display(df_pub_genre.limit(20))

publisher,genre_item,count
Big Fish Games,casual,418
Big Fish Games,adventure,392
8floor,casual,202
Choice of Games,rpg,139
Choice of Games,indie,136
HH-Games,casual,132
Laush Studio,indie,124
Choice of Games,adventure,112
Alawar Entertainment,casual,105
Sekai Project,casual,99


Databricks visualization. Run in Databricks to view.

Major publishers show clear genre preferences, Big Fish Games focuses almost exclusively on casual and adventure titles, while others such as Choice of Games or Devolver Digital specialize in indie, RPG, and adventure releases.

What are the most lucrative genres?

In [0]:
df_genre_price = (
    df_genre_full
    .where(col("price_eur").isNotNull())
    .groupBy("genre_item")
    .agg(
        round(avg("price_eur"), 2).alias("avg_price_eur"),
        count("*").alias("count_games")
    )
    .where(col("count_games") >=100)
    .orderBy(col("avg_price_eur").desc())
)

display(df_genre_price.limit(20))

genre_item,avg_price_eur,count_games
game development,21.28,159
photo editing,20.33,105
audio production,19.64,195
design & illustration,19.06,406
software training,19.03,164
video production,18.96,247
animation & modeling,18.78,322
education,14.34,317
utilities,11.45,682
simulation,9.09,10836


Non-gaming categories such as game development, photo editing, and audio production have the highest average prices, often around 20€. In contrast, typical gaming genres like casual, indie, and action titles are more affordable, generally below 10€ but widely represented.

### Platform analysis

Are most games available on Windows/Mac/Linux instead?

In [0]:
display(df_gold.limit(5))

appid,name,publisher,developer,genre,price,price_eur,discount,initialprice,initialprice_eur,release_date,release_year,required_age,positive,negative,total_ratings,per_positive_share,owners,windows,mac,linux,languages,categories,short_description,tags
10,Counter-Strike,Valve,Valve,Action,999.0,9.99,0.0,999.0,9.99,2000-11-01,2000,0,201215,5199,206414,97.5,"10,000,000 .. 20,000,000",true,true,true,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean","List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"Map(1980s -> 266, PvP -> 881, Team-Based -> 1864, 1990's -> 1191, Old School -> 769, Competitive -> 1607, Score Attack -> 289, FPS -> 4831, Tactical -> 1344, Strategy -> 614, Nostalgia -> 131, Assassin -> 227, Multiplayer -> 3392, Shooter -> 3353, First-Person -> 1707, Survival -> 304, Military -> 632, Classic -> 2784, e-sports -> 1192, Action -> 5426)"
1000000,ASCENXION,PsychoFlux Entertainment,IndigoBlue Game Studio,"Action, Adventure, Indie",999.0,9.99,0.0,999.0,9.99,2021-05-14,2021,0,27,5,32,84.4,"0 .. 20,000",true,false,false,"English, Korean, Simplified Chinese","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)","ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, to uncover the truth of this world.","Map(2D -> 159, Indie -> 69, Robots -> 100, Bullet Hell -> 179, Twin Stick Shooter -> 170, Side Scroller -> 175, Minimalist -> 136, Great Soundtrack -> 38, GameMaker -> 51, Controller -> 148, Colorful -> 124, Singleplayer -> 71, Abstract -> 111, Shooter -> 159, Difficult -> 161, Shoot 'Em Up -> 186, Metroidvania -> 181, Adventure -> 73, Atmospheric -> 88, Action -> 138)"
1000010,Crown Trick,"Team17, NEXT Studios",NEXT Studios,"Adventure, Indie, RPG, Strategy",599.0,5.99,70.0,1999.0,19.99,2020-10-16,2020,0,4032,646,4678,86.2,"200,000 .. 500,000",true,false,false,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)","Enter a labyrinth that moves as you move, where mastering the elements is key to defeating enemies and uncovering the mysteries of this underground world. With a new experience awaiting every time you enter the dungeon, let the power bestowed by the crown guide you in this challenging adventure!","Map(2D -> 205, Rogue-lite -> 226, Rogue-like -> 268, Magic -> 171, Replay Value -> 192, Fantasy -> 179, Dungeon Crawler -> 225, Perma Death -> 231, Tactical -> 184, Turn-Based -> 222, Strategy -> 225, Top-Down -> 178, RPG -> 237, Singleplayer -> 211, Difficult -> 189, Turn-Based Combat -> 254, Female Protagonist -> 217, Adventure -> 225, Action-Adventure -> 179, Turn-Based Strategy -> 216)"
1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,Vertigo Gaming Inc.,"Action, Indie, Simulation, Strategy",1999.0,19.99,0.0,1999.0,19.99,2020-10-14,2020,0,1575,115,1690,93.2,"100,000 .. 200,000",true,true,false,English,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)","Cook, serve and manage your food truck as you dish out hundreds of different foods across war-torn America in this massive sequel to the million-selling series!","Map(Family Friendly -> 175

In [0]:
df_platforms = (
    df_gold
    .agg(
        sum(col("windows").cast("int")).alias("windows_count"),
        sum(col("mac").cast("int")).alias("mac_count"),
        sum(col("linux").cast("int")).alias("linux_count")
    )
)

display(df_platforms)

windows_count,mac_count,linux_count
55676,12770,8458


Transform those three columns into a dataframe that we can plot.

In [0]:
df_platforms_unpivot = df_platforms.selectExpr(
    "stack(3, 'Windows', windows_count, 'Mac', mac_count, 'Linux', linux_count) AS (platform, count)"
)

display(df_platforms_unpivot)

platform,count
Windows,55676
Mac,12770
Linux,8458


Databricks visualization. Run in Databricks to view.

Most games are available on Windows, while Mac and Linux have much fewer supported titles.

Do certain genres tend to be preferentially available on certain platforms?

In [0]:
df_genres_platforms = (
    df_gold
    .where(col("genre").isNotNull())
    .withColumn("genre_item", explode(split(col("genre"), ",")))
    .withColumn("genre_item", trim(col("genre_item")))
    .select(
        "genre_item",
        "windows",
        "mac",
        "linux"
    )
)

In [0]:
df_genres_platform_count = (
    df_genres_platforms
    .groupBy("genre_item")
    .agg(
        sum(col("windows").cast("int")).alias("windows_count"),
        sum(col("mac").cast("int")).alias("mac_count"),
        sum(col("linux").cast("int")).alias("linux_count")
    )
    .orderBy(col("windows_count").desc())
)

display(df_genres_platform_count.limit(20))

genre_item,windows_count,mac_count,linux_count
Indie,39676,9935,6978
Action,23755,4564,3379
Casual,22082,5130,3305
Adventure,21427,5039,3302
Strategy,10892,3005,1826
Simulation,10832,2439,1532
RPG,9533,2248,1524
Early Access,6145,900,632
Free to Play,3391,845,474
Sports,2665,506,287


No, the same genres appear at the top across all platforms, so there is no clear genre that is preferentially available on one platform over another.

This exploratory analysis highlights clear patterns within the Steam ecosystem.
Game releases grew sharply after 2014 and reached record levels during the Covid period.
Windows remains the dominant platform across all genres, while Mac and Linux receive limited support.
Indie, casual, action and adventure games represent the largest share of the catalog.
Non-gaming categories such as game development or media production remain the most expensive, while traditional gaming genres tend to stay below €10.
Finally, no genre shows a strong platform preference, as the same categories dominate all operating systems.

Overall, this EDA provides a structured overview of Steam's market and can serve as a basis to identify promising genres and user trends for future game development.